In [1]:
import os
import numpy as np
import open3d as o3d

data_dir = "pcd_data/"

frames = []
pcds = []

##### manual

# tf_path = os.path.join(data_dir, "view01_tf.npy")
# tf = np.load(tf_path)
# tf_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0)
# tf_frame.transform(tf)
# frames.append(tf_frame)

##### end of manual


# ---------------------------------
# Load all tf files automatically
# ---------------------------------

tf_files = [f for f in os.listdir(data_dir) if f.endswith("_tf.npy")]
tf_files.sort()

print("Found TF files:", tf_files)

for tf_file in tf_files:

    tf_path = os.path.join(data_dir, tf_file)

    # load transformation
    tf = np.load(tf_path)

    # create coordinate frame
    tf_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0)
    tf_frame.transform(tf)
    frames.append(tf_frame)

    # create PCD
    pcd_name = tf_file.replace("_tf.npy", ".pcd")
    pcd_path = os.path.join(data_dir, pcd_name)
    pcd = o3d.io.read_point_cloud(pcd_path)
    pcds.append(pcd)
    



obj_pose_path = os.path.join(data_dir, "initial_obj_pose.npy")
tf_obj = np.load(obj_pose_path)
object_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=tf_obj[:3])
frames.append(object_frame)


world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=100.0, origin=[0, 0, 0])
frames.append(world_frame)



# Source Mesh
translation = tf_obj[:3]
rotation_deg = tf_obj[3:]
rotation_rad = np.radians(rotation_deg)
T = np.eye(4)
R = o3d.geometry.get_rotation_matrix_from_xyz(rotation_rad)
T[:3, :3] = R
T[:3, 3] = translation

SOURCE_PATH = "workpiece/first_object.stl" #  CAD model
mesh = o3d.io.read_triangle_mesh(SOURCE_PATH)
mesh.compute_vertex_normals() # Makes the shading look better
R_mat = mesh.get_rotation_matrix_from_axis_angle([0, 0, np.radians(90)])
mesh.rotate(R_mat, center=[0, 0, 0])
mesh.transform(T)


# pcd_path = os.path.join(data_dir, "view01.pcd")
# pcd = o3d.io.read_point_cloud(pcd_path)


o3d.visualization.draw_geometries(frames + pcds)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Found TF files: ['view00_tf.npy', 'view01_tf.npy', 'view02_tf.npy', 'view03_tf.npy', 'view04_tf.npy', 'view05_tf.npy', 'view06_tf.npy', 'view07_tf.npy', 'view08_tf.npy', 'view09_tf.npy', 'view10_tf.npy', 'view11_tf.npy', 'view12_tf.npy', 'view13_tf.npy', 'view14_tf.npy', 'view15_tf.npy', 'view16_tf.npy', 'view17_tf.npy', 'view18_tf.npy', 'view19_tf.npy']
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view17.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view18.pcd
[Open3D WARNING] Read PCD failed: unable to open file: pcd_data\view19.pcd
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding box.
[Open3D WARNING] The number of points is 0 when creating axis-aligned bounding bo

In [7]:
# Helper Functions

import open3d as o3d
import numpy as np
import math
import copy
import os


def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0], 
                     [s,  c, 0], 
                     [0,  0, 1]])


def merge_multiview_scan(data_dir, initial_pos, initial_angle, box_size):
    """Merges multiple view PCDs and removes points below the detected plane."""
    pcd_files = sorted([f for f in os.listdir(data_dir) if f.endswith('.pcd') and "view0" in f])
    merged_pcd = o3d.geometry.PointCloud()
    
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos), 
        R=get_rotation_matrix_z(-initial_angle), 
        extent=np.array(box_size)
    )

    for file_name in pcd_files:
        pcd = o3d.io.read_point_cloud(os.path.join(data_dir, file_name))
   
        # 1. Detect the plane
        plane_model, inliers = pcd.segment_plane(distance_threshold=3.0, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model

        # 2. Extract all points as a numpy array
        pts = np.asarray(pcd.points)

        # 3. Calculate distance to plane for every point: ax + by + cz + d
        # Points on the plane result in 0. Points above are positive, below are negative.
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        
        # 4. Create an index of points that are ABOVE the plane
        # A small offset (e.g., 0.5mm) to avoid keeping table noise
        above_plane_indices = np.where(distances > 0.5)[0]
        pcd = pcd.select_by_index(above_plane_indices)

        # 5. Crop to the OBB and merge
        merged_pcd += pcd.crop(obb)

    return merged_pcd


def preprocess_normal(pcd, num_points=False, invert_normals=False):
    """Downsamples, estimates normals, and computes FPFH features."""
    if num_points:
        pcd_down = pcd.farthest_point_down_sample(num_points)
    else:
        pcd_down = pcd
    avg_dist = np.mean(pcd_down.compute_nearest_neighbor_distance())
    
    # Estimate Normals
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 2, max_nn=30))
    
    # Orientation fix: Ensure normals point 'up' (positive Z)
    normals = np.asarray(pcd_down.normals)
    
    if invert_normals:
        for i in range(len(normals)):
            if normals[i][2] < 0:
                normals[i] *= -1

    # Compute FPFH (Feature descriptors for global matching)
    fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=avg_dist * 5, max_nn=100))
    
    return pcd_down, fpfh


def run_global_registration(source, target, source_fpfh, target_fpfh, voxel_size):
    """
    Performs RANSAC-based global registration to find a rough alignment.
    """
    distance_threshold = voxel_size * 1.5
    
    # RANSAC based on feature matching
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source, target, source_fpfh, target_fpfh, 
        mutual_filter=True,
        max_correspondence_distance=distance_threshold,
        estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        ransac_n=3, 
        checkers=[
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
        ], 
        criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(4000000, 500)
    )
    return result


def run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh):
    """
    Performs iterative RANSAC-based global registration to find the best rough alignment.
    Matches the logic of testing multiple thresholds to find the highest fitness and lowest RMSE.
    """
    max_attempts = 5
    best_fitness = -0.1
    best_inlier_rmse = 100.0
    best_result = None
    best_threshold = None 

    # Thresholds to test, from coarse (10.0) to fine (3.0)
    thresholds = [10.0, 9.0, 8.0, 7.0, 6.0, 5.0, 4.0, 3.0]
    # thresholds = [7]

    for attempt in range(max_attempts):
        for thr in thresholds:            
            result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
                source_down, target_down, source_fpfh, target_fpfh, 
                mutual_filter=True, # Improved matching
                max_correspondence_distance=thr,
                estimation_method=o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
                ransac_n=3, 
                checkers=[
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85),
                    o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(thr)
                ], 
                criteria=o3d.pipelines.registration.RANSACConvergenceCriteria(10000, 0.99)
            )

            # Update best result if fitness is good and RMSE is lower
            if result.fitness > 0.85 and result.inlier_rmse < best_inlier_rmse:
                best_fitness = result.fitness
                best_inlier_rmse = result.inlier_rmse
                best_result = result
                best_threshold = thr
            
            # Early exit if we find a very high-quality match
            if best_fitness > 0.95 and best_inlier_rmse < 2.35: 
                print(f"Excellent Global Fit Found at Threshold {best_threshold}")
                return best_result, best_threshold
            
    if best_result is None:
        print("Warning: RANSAC could not find a fit above 0.85 fitness.")
        # Fallback to the last result generated if nothing met the 0.85 criteria
        return result, thr

    print(f"RANSAC Finished. Best Threshold: {best_threshold} | Fitness: {best_fitness:.4f}")
    return best_result, best_threshold


# One time local refinement using ICP
def run_local_refinement(source, target, initial_transformation, voxel_size):
    """
    Performs ICP registration to refine the alignment found by RANSAC.
    """
    # We use a smaller threshold for ICP to ensure high precision
    distance_threshold = voxel_size * 0.4
    
    result = o3d.pipelines.registration.registration_icp(
        source, target, distance_threshold, initial_transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000)
    )
    return result


# Progressive Local Refinement using ICP
def run_local_refinement_adaptive(source, target, initial_trans, best_ransac_thr):
    """
    Refines alignment using an iterative ICP loop. 
    Starts with a coarse threshold and progressively tightens for precision.
    """
    # Define a range of multipliers to tighten the search radius
    # multipliers = [1.0, 0.8, 0.5, 0.25]
    multipliers = [1.0, 0.9, 0.8, 0.7, 0.6, 0.5, 0.4, 0.3, 0.2, 0.1]
    # multipliers = [0.6]
    thresholds = [best_ransac_thr * m for m in multipliers]
    
    best_result = None
    best_inlier_rmse = float('inf') # Start with the highest possible error
    
    print(f"{'Threshold':<12} | {'Fitness':<12} | {'RMSE':<12}")
    print("-" * 45)

    for thr in thresholds:
        # Execute Point-to-Point ICP
        reg_icp = o3d.pipelines.registration.registration_icp(
            source, target, thr, initial_trans,
            o3d.pipelines.registration.TransformationEstimationPointToPoint(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1000)
        )

        # Logging the current iteration results
        print(f"{thr:<12.2f} | {reg_icp.fitness:<12.4f} | {reg_icp.inlier_rmse:<12.4f}")

        # Selection Logic: Prioritize lowest RMSE (highest precision) 
        # as long as the fitness is acceptable (e.g., > 85%)
        if reg_icp.fitness > 0.85 and reg_icp.inlier_rmse < best_inlier_rmse:
            best_inlier_rmse = reg_icp.inlier_rmse
            best_result = reg_icp

    # Fallback: if no run met the 85% fitness, return the last result
    return best_result if best_result is not None else reg_icp

In [19]:
# Main Processing Pipeline

# --- 1. Configuration ---
DATA_DIR = "pcd_data"
SOURCE_PATH = "workpiece/first_object_10000.pcd" #  CAD model

tf_obj = np.load(r'pcd_data/initial_obj_pose.npy')
YOLO_POS = tf_obj[:3]                                   # Initial guess from YOLO
YOLO_ANGLE = tf_obj[4]                                  # Initial angle from YOLO
CROP_BOX = [85, 65, 80]                               # Region of interest size


# --- 2. Data Preparation ---
print("Step 1: Merging and cleaning scans...")
source_cloud = o3d.io.read_point_cloud(SOURCE_PATH)
target_cloud = merge_multiview_scan(DATA_DIR, YOLO_POS, YOLO_ANGLE, CROP_BOX)
o3d.visualization.draw_geometries([target_cloud], window_name="Merged Target Cloud")

# Get downsampled versions and features for RANSAC
source_down, source_fpfh = preprocess_normal(source_cloud, 10000)
target_down, target_fpfh = preprocess_normal(target_cloud, 10000, invert_normals=True)
o3d.visualization.draw_geometries([source_down, target_down], window_name="Downsampled Source Cloud")
# o3d.visualization.draw_geometries([target_down], window_name="Downsampled Target Cloud")

Step 1: Merging and cleaning scans...


In [20]:
# --- 3. Global Alignment (RANSAC) ---
print("Step 2: Running RANSAC Global Registration...")
ransac_res, best_thr = run_global_registration_adaptive(source_down, target_down, source_fpfh, target_fpfh)
print(ransac_res)

# Visualize RANSAC result
source_temp = copy.deepcopy(source_down)
source_temp.transform(ransac_res.transformation)
source_temp.paint_uniform_color([1, 0, 0])
target_down.paint_uniform_color([0, 0.651, 0.929])
o3d.visualization.draw_geometries([source_temp, target_down], window_name="RANSAC Result")

Step 2: Running RANSAC Global Registration...
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (205) after mutual filter, fall back to original correspo

In [23]:
# --- 4. Local Alignment (ICP) ---
print("Step 3: Running ICP Local Refinement...")
icp_res = run_local_refinement_adaptive(source_down, target_down, ransac_res.transformation, best_thr)
print(icp_res)

# --- 5. Extract Final Results ---
final_matrix = icp_res.transformation
initial_POS = final_matrix[:3, 3] 
initial_ORI = final_matrix[:3, :3] 
print("="*30)
print(f"Final Position: {initial_POS}")
print(f"Final Orientation Matrix:\n{initial_ORI}")
print(f"Fitness: {icp_res.fitness:.4f}")
print(f"RMSE: {icp_res.inlier_rmse:.4f}")


# --- 6. Final Visualization ---
source_final = copy.deepcopy(source_down).transform(final_matrix)
source_final.paint_uniform_color([1, 0, 0])         # Red: CAD Model
target_cloud.paint_uniform_color([0, 0.65, 0.93])   # Blue: Scanned Data
o3d.visualization.draw_geometries([source_final, target_cloud])

Step 3: Running ICP Local Refinement...
Threshold    | Fitness      | RMSE        
---------------------------------------------
10.00        | 1.0000       | 2.4254      
9.00         | 0.9996       | 2.4189      
8.00         | 0.9985       | 2.4036      
7.00         | 0.9923       | 2.3384      
6.00         | 0.9787       | 2.2294      
5.00         | 0.9542       | 2.0807      
4.00         | 0.9068       | 1.8758      
3.00         | 0.8032       | 1.5587      
2.00         | 0.6048       | 1.1546      
1.00         | 0.1573       | 0.7524      
RegistrationResult with fitness=9.068000e-01, inlier_rmse=1.875839e+00, and correspondence_set size of 9068
Access transformation to get result.
Final Position: [559.08610883 -62.24074333  -2.49356033]
Final Orientation Matrix:
[[-0.43622285  0.89971918 -0.01466347]
 [-0.89982749 -0.43607828  0.01209258]
 [ 0.0044855   0.01846965  0.99981936]]
Fitness: 0.9068
RMSE: 1.8758
